## Ingestion - 1

This notebook will be used to ingest data knowledge base data into a vector index for similarity search in an offline manner. 

### Plan

1. Using sentence transfoermers (SBERT) to use pre-trained models to generate embeddings for the documents

1. The embeddings are then stored in a FAISS index for efficient similarity search.

In [15]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer


In [16]:
import faiss
import numpy as np
import json
import os

In [17]:
NUM_DOCS = 50

ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

docs = []
for i, row in enumerate(ds):
    if i >= NUM_DOCS:
        break
    docs.append({"id": row["id"], "title": row["title"], "text": row["text"], "url": row["url"]})

print(f"Loaded {len(docs)} documents")
print(f"Sample title: {docs[0]['title']}")
print(f"Sample text length: {len(docs[0]['text'])} chars")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loaded 50 documents
Sample title: Anarchism
Sample text length: 46064 chars


In [18]:
MAX_CHUNK_CHARS = 4000
OVERLAP_CHARS = 200

chunks = []
for doc in docs:
    text = doc["text"]
    start = 0
    chunk_idx = 0
    while start < len(text):
        end = min(start + MAX_CHUNK_CHARS, len(text))
        chunk_text = text[start:end]
        chunks.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "url": doc["url"],
            "chunk_idx": chunk_idx,
            "text": chunk_text,
        })
        start += MAX_CHUNK_CHARS - OVERLAP_CHARS
        chunk_idx += 1

print(f"Created {len(chunks)} chunks from {len(docs)} documents")

Created 454 chunks from 50 documents


In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
embeddings = model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings shape: (454, 384)


In [20]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")

FAISS index built with 454 vectors of dimension 384


In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "faiss_store")
os.makedirs(DATA_DIR, exist_ok=True)

faiss.write_index(index, os.path.join(DATA_DIR, "wiki_index.faiss"))

with open(os.path.join(DATA_DIR, "wiki_chunks.json"), "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved FAISS index and {len(chunks)} chunk metadata to {DATA_DIR}")

Saved FAISS index and 454 chunk metadata to c:\Users\satwikide\repo\RAG-with-evals-samples\RAG on documents\data


In [23]:
query = "history of mathematics"
query_vec = model.encode([query], convert_to_numpy=True).astype(np.float32)

k = 5
distances, indices = index.search(query_vec, k)

print(f"Query: '{query}'\n")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    c = chunks[idx]
    print(f"  [{rank}] (dist={dist:.4f}) {c['title']} — chunk {c['chunk_idx']}")
    print(f"      {c['text'][:120]}...\n")

Query: 'history of mathematics'

  [1] (dist=0.9967) Apollo — chunk 6
      ts were considered divine, and their forms were preserved in the marble or stone elements of the temples of Doric order....

  [2] (dist=1.1781) Alchemy — chunk 9
      ry/research institute. Michael Sendivogius (Michał Sędziwój, 1566–1636), a Polish alchemist, philosopher, medical doctor...

  [3] (dist=1.1943) Alchemy — chunk 6
       encouraged rationalism in a Christian context. In the early 12th century, Peter Abelard followed Anselm's work, laying ...

  [4] (dist=1.2951) Alain Connes — chunk 0
      Alain Connes (; born 1 April 1947 in Draguignan) is a French mathematician, known for his contributions to the study of ...

  [5] (dist=1.3052) Alchemy — chunk 15
       in the Classical World, Oxford University Press, 2018, p. 409–430.
 Jean Letrouit, "Chronologie des alchimistes grecs,"...

